<a href="https://colab.research.google.com/github/marginagonzales1992-png/Mis-Practicas/blob/main/Practica_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import simpy
import random

# Parámetros del sistema
TIEMPO_MOLDEO = 30        # segundos (media)
TIEMPO_ENFRIAMIENTO = 60  # segundos (constante)
TIEMPO_INSPECCION = 45    # segundos (media)
TAMANO_LOTE_EMPAQUE = 100 # botellas por lote
NUM_MAQUINAS_MOLDEO = 2
NUM_INSPECTORES = 1
PROB_DEFECTO = 0.15        # probabilidad de que una botella sea defectuosa

# Proceso de producción de una botella
def producir_botella(env, id_botella, moldeo, inspeccion, empaque, descartadas):
    # Moldeo
    with moldeo.request() as req:
        yield req
        tiempo = random.normalvariate(TIEMPO_MOLDEO, 5)
        yield env.timeout(tiempo)
        print(f"Botella {id_botella} moldeada en {env.now:.1f}")

    # Enfriamiento
    yield env.timeout(TIEMPO_ENFRIAMIENTO)
    print(f"Botella {id_botella} enfriada en {env.now:.1f}")

    # Inspección
    with inspeccion.request() as req:
        yield req
        tiempo = random.expovariate(1.0/TIEMPO_INSPECCION)
        yield env.timeout(tiempo)

        # Decisión: ¿aprobada o defectuosa?
        if random.random() < PROB_DEFECTO:
            descartadas.append(id_botella)
            print(f"❌ Botella {id_botella} descartada en {env.now:.1f}")
            return
        else:
            print(f"✅ Botella {id_botella} aprobada en {env.now:.1f}")

    # Empaque (se acumulan hasta formar un lote)
    empaque.append(id_botella)
    if len(empaque) >= TAMANO_LOTE_EMPAQUE:
        print(f"📦 Lote de {TAMANO_LOTE_EMPAQUE} botellas empacado en {env.now:.1f}")
        empaque.clear()

# Generador de llegadas de botellas
def llegada_botellas(env, moldeo, inspeccion, empaque, descartadas):
    id_botella = 0
    while True:
        yield env.timeout(random.expovariate(1/20))  # llegada cada ~20 seg
        id_botella += 1
        env.process(producir_botella(env, id_botella, moldeo, inspeccion, empaque, descartadas))

# Configuración del entorno
env = simpy.Environment()
moldeo = simpy.Resource(env, NUM_MAQUINAS_MOLDEO)
inspeccion = simpy.Resource(env, NUM_INSPECTORES)
empaque = []
descartadas = []

# Iniciar simulación
env.process(llegada_botellas(env, moldeo, inspeccion, empaque, descartadas))
env.run(until=500)  # simular 500 segundos

# Resumen final
print("\n--- RESULTADOS ---")
print(f"Botellas descartadas: {len(descartadas)}")
print(f"IDs descartados: {descartadas}")